# Carbon-Aware, Privacy-Preserving, Thermal-Active DNN Split Scheduling — Simulation Suite

This notebook merges four simulation scripts into one runnable sequence, each building on the previous section's definitions. Run the cells in order from top to bottom.

1. **Core Simulation Model** — latency, carbon, privacy, and active thermal-battery conduction cost model, with a sustained-load thermal throttle experiment.
2. **Adaptive Weight Controller (AWC) Evaluation** — an online multiplicative-feedback controller compared against a static weight preset under a combined carbon-spike / thermal-excursion scenario.
3. **Cross-Architecture Generalisation** — extends the cost model to ResNet-50, ViT-Base, and a multimodal profile to test whether the framework's design generalises beyond MobileNetV2.
4. **Comparison Against SOTA-Adjacent Baselines** — adds a carbon-aware-greedy and a privacy-aware-static baseline and compares all five strategies head to head.

All per-layer power, timing, and tensor-size figures used throughout are documented modelling assumptions, not hardware measurements — this is a simulation study.

## 1. Core Simulation Model

Defines the edge-cloud split cost model — latency, carbon emissions, a privacy penalty from normalised mutual-information decay, and an active thermal-battery conduction model that forecasts device temperature one control interval ahead for each candidate split (rather than reacting after the fact). Includes a DVFS backstop and a dynamic weight-escalation mechanism, and the split-selection optimiser used by every later section.

Also runs the sustained-load thermal throttle experiment (50 trials, 60-second continuous inference stream per trial) comparing local-only, latency-first, and the proposed active strategy.

In [1]:
import numpy as np

rng = np.random.default_rng(42)

L = 18
PRIV_ZONE = {0, 1, 2, 3}

_raw = np.array([1.6, 1.5, 1.4, 1.3, 1.2, 1.15, 1.1, 1.05, 1.0, 0.95,
                  0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55])
t_edge_ms = _raw / _raw.sum() * 450.0

P_edge_layer_W = 4.0 + rng.normal(0, 0.15, L)

tensor_kb = np.array([588, 588, 294, 294, 147, 147, 73.5, 73.5,
                       36.8, 36.8, 18.4, 18.4, 9.2, 9.2, 4.6, 4.6, 2.3, 2.3])

layer_mflops = np.array([54, 54, 44, 44, 35, 35, 28, 28, 22, 22,
                          18, 18, 15, 15, 12, 12, 10, 10])

E_per_bit_nJ = 1.0
CLOUD_MFLOPS_PER_MS = 500.0
J_PER_MFLOP_CLOUD = 0.00003

regions = {
    'IN-South': dict(CI_now=720, CI_fcast=680, rtt_ms=20,  compute_scale=1.0),
    'EU-West':  dict(CI_now=180, CI_fcast=140, rtt_ms=180, compute_scale=1.0),
    'US-East':  dict(CI_now=410, CI_fcast=390, rtt_ms=240, compute_scale=1.0),
}
PUE = 1.2

C_TH = 3.0
R_TH = 7.0
T_SAFE = 35.0
T_CRITICAL = 42.0
DT_FORECAST_S = 1.0

R0, R1, R2, K_SOC = 0.05, 0.15, 0.02, 4.0
T_REF = 25.0
I_DISCHARGE_A = 1.2
V_crit = 3.0

def r_int(soc, t_device):
    return R0 + R1 * np.exp(-K_SOC * soc) + R2 * max(0.0, T_REF - t_device)

def p_batt(soc, t_device):
    return I_DISCHARGE_A ** 2 * r_int(soc, t_device)

def p_compute(k):
    return float(np.mean(P_edge_layer_W[:k + 1]))

def mutual_info(k):
    return np.exp(-0.35 * k)

_I_MAX = mutual_info(0)
_I_MIN = mutual_info(L - 1)

def P_priv(k):
    return (mutual_info(k) - _I_MIN) / (_I_MAX - _I_MIN)

def edge_time_ms(k):
    return float(np.sum(t_edge_ms[:k + 1]))

def edge_energy_J(k):
    idx = slice(0, k + 1)
    return float(np.sum(P_edge_layer_W[idx] * (t_edge_ms[idx] / 1000.0)))

def uplink_time_ms(k, region, bandwidth_mbps):
    bits = tensor_kb[k] * 1024 * 8
    transmit_ms = bits / (bandwidth_mbps * 1e6) * 1000.0
    return transmit_ms + region['rtt_ms'] / 2.0

def net_energy_J(k):
    bits = tensor_kb[k] * 1024 * 8
    return bits * E_per_bit_nJ * 1e-9

def cloud_time_ms(k, region):
    remaining = float(np.sum(layer_mflops[k + 1:])) if k + 1 < L else 0.0
    return remaining / (CLOUD_MFLOPS_PER_MS * region['compute_scale'])

def cloud_energy_J(k, region):
    remaining = float(np.sum(layer_mflops[k + 1:])) if k + 1 < L else 0.0
    return remaining * J_PER_MFLOP_CLOUD * PUE

def latency_ms(k, region_name, bandwidth_mbps):
    region = regions[region_name]
    return (edge_time_ms(k) + uplink_time_ms(k, region, bandwidth_mbps)
            + cloud_time_ms(k, region) + 2.0)

def carbon_gco2(k, region_name, bandwidth_mbps):
    region = regions[region_name]
    e_edge = edge_energy_J(k)
    e_net = net_energy_J(k)
    e_cloud = cloud_energy_J(k, region)
    e_kwh = (e_edge + e_net + e_cloud) / 3.6e6
    return e_kwh * region['CI_now']

def forecast_temp(k, t_device, t_amb, soc):
    p_gen = p_compute(k) + p_batt(soc, t_device)
    dTdt = (p_gen - (t_device - t_amb) / R_TH) / C_TH
    return t_device + DT_FORECAST_S * dTdt

def thermal_penalty(k, t_device, t_amb, soc):
    s_hat = forecast_temp(k, t_device, t_amb, soc)
    return max(0.0, s_hat - T_SAFE) ** 2

WEIGHTS_BASE = dict(lat=0.15, carb=0.55, priv=0.15, therm=0.15)
THERM_ESCALATION_MARGIN_C = 4.0
THERM_ESCALATION_CAP = 0.80
DVFS_POWER_MULTIPLIER = 0.8 ** 3

def dynamic_weights(t_device):
    excess = np.clip((t_device - (T_SAFE - THERM_ESCALATION_MARGIN_C))
                      / THERM_ESCALATION_MARGIN_C, 0.0, 1.0)
    w_therm = WEIGHTS_BASE['therm'] + excess * (THERM_ESCALATION_CAP - WEIGHTS_BASE['therm'])
    remaining = 1.0 - w_therm
    scale = remaining / (1.0 - WEIGHTS_BASE['therm'])
    return dict(lat=WEIGHTS_BASE['lat'] * scale,
                carb=WEIGHTS_BASE['carb'] * scale,
                priv=WEIGHTS_BASE['priv'] * scale,
                therm=w_therm)

def normalise(values):
    vmin, vmax = min(values), max(values)
    span = vmax - vmin
    if span < 1e-9:
        return [0.0 for _ in values]
    return [(v - vmin) / span for v in values]

def select_split_active(bandwidth_mbps, t_device, t_amb, soc):
    weights = dynamic_weights(t_device)
    feasible = [(k, rname) for k in range(L) if k not in PRIV_ZONE
                for rname in regions]
    raw_lat = [latency_ms(k, r, bandwidth_mbps) for k, r in feasible]
    raw_carb = [carbon_gco2(k, r, bandwidth_mbps) for k, r in feasible]
    raw_therm = [thermal_penalty(k, t_device, t_amb, soc) for k, r in feasible]
    n_lat = normalise(raw_lat)
    n_carb = normalise(raw_carb)
    n_therm = normalise(raw_therm)

    best = None
    best_J = float('inf')
    for i, (k, r) in enumerate(feasible):
        J = (weights['lat'] * n_lat[i] + weights['carb'] * n_carb[i]
             + weights['priv'] * P_priv(k) + weights['therm'] * n_therm[i])
        if J < best_J:
            best_J = J
            best = (k, r)

    k_sel, r_sel = best
    s_hat = forecast_temp(k_sel, t_device, t_amb, soc)
    dvfs_multiplier = DVFS_POWER_MULTIPLIER if s_hat >= T_SAFE else 1.0
    return k_sel, r_sel, dvfs_multiplier

def select_split_latency_first(bandwidth_mbps):
    feasible = [(k, rname) for k in range(L) for rname in regions]
    best = min(feasible, key=lambda kr: latency_ms(kr[0], kr[1], bandwidth_mbps))
    return best

P_IDLE_W = 1.5
SIM_DURATION_S = 60.0
STEP_S = 0.05

def _integrate_step(t_device, soc, p_active, t_amb, dt):
    p_gen = p_active + p_batt(soc, t_device)
    dTdt = (p_gen - (t_device - t_amb) / R_TH) / C_TH
    return t_device + dt * dTdt

def simulate_sustained(strategy, t_init, t_amb, soc, bandwidth_mbps):
    t_device = t_init
    elapsed = 0.0
    throttled = False
    max_temp = t_device
    k_history = []
    carbon_samples = []
    latency_samples = []

    if strategy == 'latency_first':
        k_fixed, r_fixed = select_split_latency_first(bandwidth_mbps)

    while elapsed < SIM_DURATION_S and not throttled:
        dvfs_mult = 1.0
        v_bat = 3.0 + 0.7 * soc
        if v_bat <= V_crit:
            k, r = 0, min(regions, key=lambda rn: latency_ms(0, rn, bandwidth_mbps))
        elif strategy == 'local_only':
            k, r = L - 1, None
        elif strategy == 'latency_first':
            k, r = k_fixed, r_fixed
        elif strategy == 'proposed_active':
            k, r, dvfs_mult = select_split_active(bandwidth_mbps, t_device, t_amb, soc)
        else:
            raise ValueError(strategy)

        k_history.append(k)
        if r is not None:
            carbon_samples.append(carbon_gco2(k, r, bandwidth_mbps))
            latency_samples.append(latency_ms(k, r, bandwidth_mbps))
            cycle_ms = latency_ms(k, r, bandwidth_mbps)
        else:
            carbon_samples.append(carbon_gco2(k, 'IN-South', bandwidth_mbps))
            latency_samples.append(edge_time_ms(k))
            cycle_ms = edge_time_ms(k)

        active_s = edge_time_ms(k) / 1000.0
        idle_s = max(0.0, cycle_ms / 1000.0 - active_s)
        active_power = p_compute(k) * dvfs_mult

        t_remaining = active_s
        while t_remaining > 0 and not throttled:
            dt = min(STEP_S, t_remaining)
            t_device = _integrate_step(t_device, soc, active_power, t_amb, dt)
            max_temp = max(max_temp, t_device)
            if t_device >= T_CRITICAL:
                throttled = True
            t_remaining -= dt
            elapsed += dt
            if elapsed >= SIM_DURATION_S:
                break

        t_remaining = idle_s
        while t_remaining > 0 and not throttled and elapsed < SIM_DURATION_S:
            dt = min(STEP_S, t_remaining)
            t_device = _integrate_step(t_device, soc, P_IDLE_W, t_amb, dt)
            max_temp = max(max_temp, t_device)
            if t_device >= T_CRITICAL:
                throttled = True
            t_remaining -= dt
            elapsed += dt

    return dict(throttled=throttled, max_temp=max_temp, final_temp=t_device,
                k_history=k_history, carbon_samples=carbon_samples,
                latency_samples=latency_samples)

N_TRIALS = 50

def run_experiment():
    results = {name: dict(throttle=0, temps=[], k_chosen=[], carbon=[], latency=[])
               for name in ['local_only', 'latency_first', 'proposed_active']}

    for trial in range(N_TRIALS):
        bandwidth = float(rng.lognormal(mean=np.log(10.0), sigma=0.5))
        t_init = float(rng.uniform(28, 36))
        t_amb = float(rng.uniform(22, 30))
        soc = float(rng.uniform(0.2, 1.0))

        for name in ['local_only', 'latency_first', 'proposed_active']:
            out = simulate_sustained(name, t_init, t_amb, soc, bandwidth)
            results[name]['throttle'] += int(out['throttled'])
            results[name]['temps'].append(out['max_temp'])
            results[name]['k_chosen'].extend(out['k_history'])
            results[name]['carbon'].append(float(np.mean(out['carbon_samples'])))
            results[name]['latency'].append(float(np.mean(out['latency_samples'])))

    return results

if __name__ == '__main__':
    res = run_experiment()
    local_carbon_ref = float(np.mean(res['local_only']['carbon']))

    print(f"{'Strategy':<18}{'Throttle%':>10}{'MeanTemp':>10}{'MaxTemp':>10}"
          f"{'MeanLat(ms)':>13}{'Carbon(norm%)':>15}{'ModeSplit':>11}")
    for name in ['local_only', 'latency_first', 'proposed_active']:
        r = res[name]
        throttle_pct = 100.0 * r['throttle'] / N_TRIALS
        mean_temp = np.mean(r['temps'])
        max_temp = np.max(r['temps'])
        mean_lat = np.mean(r['latency'])
        mean_carbon = np.mean(r['carbon'])
        norm_carbon = 100.0 * mean_carbon / local_carbon_ref
        mode_k = int(np.bincount(r['k_chosen']).argmax())
        print(f"{name:<18}{throttle_pct:>10.1f}{mean_temp:>10.2f}{max_temp:>10.2f}"
              f"{mean_lat:>13.1f}{norm_carbon:>15.1f}{mode_k:>11d}")

    print()
    print("Split-point distribution (proposed_active):",
          dict(zip(*np.unique(res['proposed_active']['k_chosen'], return_counts=True))))
    print("Split-point distribution (latency_first):",
          dict(zip(*np.unique(res['latency_first']['k_chosen'], return_counts=True))))

Strategy           Throttle%  MeanTemp   MaxTemp  MeanLat(ms)  Carbon(norm%)  ModeSplit
local_only             100.0     42.02     42.04        450.0          100.0         17
latency_first          100.0     42.01     42.03        300.9           46.3          4
proposed_active          2.0     38.23     42.00        426.6           11.3          5

Split-point distribution (proposed_active): {np.int64(4): np.int64(341), np.int64(5): np.int64(6636), np.int64(6): np.int64(206), np.int64(8): np.int64(3), np.int64(9): np.int64(1)}
Split-point distribution (latency_first): {np.int64(2): np.int64(445), np.int64(4): np.int64(1901), np.int64(6): np.int64(1037), np.int64(8): np.int64(369)}


## 2. Adaptive Weight Controller (AWC) Evaluation

Evaluates a feedback-driven online controller that adjusts the four objective weights via multiplicative updates against operator-specified per-objective targets, rather than using a single fixed weight vector. Compared against the best static preset over a 200-cycle scenario that combines a carbon-intensity spike in the normally-lowest-carbon region with a concurrent thermal excursion, tracking cumulative target violation for each controller.

In [2]:
import numpy as np

rng = np.random.default_rng(7)

N_CYCLES = 200
TAU = dict(lat=0.30, carb=0.30, priv=0.30, therm=0.30)
ETA = 3.0
W_MIN, W_MAX = 0.05, 0.70
STATIC_W = dict(lat=0.15, carb=0.55, priv=0.15, therm=0.15)

EU_WEST_CI_NORMAL = regions['EU-West']['CI_now']
EU_WEST_CI_SPIKE = 650

def scenario_conditions(cycle):
    if 70 <= cycle < 110:
        regions['EU-West']['CI_now'] = EU_WEST_CI_SPIKE
    else:
        regions['EU-West']['CI_now'] = EU_WEST_CI_NORMAL
    t_amb = 34.0 if 120 <= cycle < 160 else 24.0
    return t_amb

def normalise_costs(bandwidth_mbps, t_device, t_amb, soc):
    feasible = [(k, rname) for k in range(L) if k not in PRIV_ZONE
                for rname in regions]
    raw_lat = [latency_ms(k, r, bandwidth_mbps) for k, r in feasible]
    raw_carb = [carbon_gco2(k, r, bandwidth_mbps) for k, r in feasible]
    raw_therm = [thermal_penalty(k, t_device, t_amb, soc) for k, r in feasible]
    n_lat = normalise(raw_lat)
    n_carb = normalise(raw_carb)
    n_therm = normalise(raw_therm)
    return feasible, n_lat, n_carb, n_therm

def choose_split(weights, bandwidth_mbps, t_device, t_amb, soc):
    feasible, n_lat, n_carb, n_therm = normalise_costs(bandwidth_mbps, t_device, t_amb, soc)
    best, best_J = None, float('inf')
    best_costs = None
    for i, (k, r) in enumerate(feasible):
        priv = P_priv(k)
        J = (weights['lat'] * n_lat[i] + weights['carb'] * n_carb[i]
             + weights['priv'] * priv + weights['therm'] * n_therm[i])
        if J < best_J:
            best_J = J
            best = (k, r)
            best_costs = dict(lat=n_lat[i], carb=n_carb[i], priv=priv, therm=n_therm[i])
    return best, best_costs

def mwu_update(weights, costs):
    updated = {}
    total = 0.0
    for key in weights:
        excess = max(0.0, costs[key] - TAU[key])
        w_new = weights[key] * np.exp(ETA * excess)
        updated[key] = w_new
        total += w_new
    for key in updated:
        updated[key] = np.clip(updated[key] / total, W_MIN, W_MAX)
    z = sum(updated.values())
    for key in updated:
        updated[key] /= z
    return updated

def run(controller):
    t_device = 30.0
    soc = 0.7
    weights = dict(STATIC_W)
    total_violation = 0.0
    per_obj_violation = {k: 0.0 for k in TAU}
    per_obj_violation_count = {k: 0 for k in TAU}
    weight_trace = []

    for cycle in range(N_CYCLES):
        t_amb = scenario_conditions(cycle)
        bandwidth = float(rng.lognormal(mean=np.log(10.0), sigma=0.3))
        (k, r), costs = choose_split(weights, bandwidth, t_device, t_amb, soc)

        p_gen = p_compute(k) + p_batt(soc, t_device)
        dTdt = (p_gen - (t_device - t_amb) / R_TH) / C_TH
        t_device = t_device + DT_FORECAST_S * dTdt

        for key in TAU:
            excess = max(0.0, costs[key] - TAU[key])
            total_violation += excess
            per_obj_violation[key] += excess
            if excess > 0:
                per_obj_violation_count[key] += 1

        if controller == 'adaptive':
            weights = mwu_update(weights, costs)
        weight_trace.append(dict(weights))

    return dict(total_violation=total_violation,
                per_obj_violation=per_obj_violation,
                per_obj_violation_count=per_obj_violation_count,
                weight_trace=weight_trace)

if __name__ == '__main__':
    static_res = run('static')
    regions['EU-West']['CI_now'] = EU_WEST_CI_NORMAL
    rng2 = np.random.default_rng(7)
    globals()['rng'] = rng2
    adaptive_res = run('adaptive')

    print(f"{'Controller':<12}{'TotalViolation':>16}{'lat':>8}{'carb':>8}{'priv':>8}{'therm':>8}")
    for name, res in [('static', static_res), ('adaptive', adaptive_res)]:
        pv = res['per_obj_violation']
        print(f"{name:<12}{res['total_violation']:>16.2f}"
              f"{pv['lat']:>8.2f}{pv['carb']:>8.2f}{pv['priv']:>8.2f}{pv['therm']:>8.2f}")

    print(f"\n{'Controller':<12}{'ViolCycles/200':>16}{'lat':>8}{'carb':>8}{'priv':>8}{'therm':>8}")
    for name, res in [('static', static_res), ('adaptive', adaptive_res)]:
        pc = res['per_obj_violation_count']
        total_cycles_any = sum(1 for c in range(N_CYCLES))
        print(f"{name:<12}{'':>16}{pc['lat']:>8d}{pc['carb']:>8d}{pc['priv']:>8d}{pc['therm']:>8d}")

    reduction = 100.0 * (1 - adaptive_res['total_violation'] / static_res['total_violation'])
    print(f"\nTotal violation reduction (adaptive vs static): {reduction:.1f}%")

    for cyc in [0, 69, 90, 119, 140, 159, 199]:
        w = adaptive_res['weight_trace'][cyc]
        print(f"cycle {cyc:3d}: lat={w['lat']:.3f} carb={w['carb']:.3f} "
              f"priv={w['priv']:.3f} therm={w['therm']:.3f}")

Controller    TotalViolation     lat    carb    priv   therm
static                 25.07   25.07    0.00    0.00    0.00
adaptive                7.03    2.70    2.13    0.00    2.20

Controller    ViolCycles/200     lat    carb    priv   therm
static                           198       0       0       0
adaptive                          97      21       0      12

Total violation reduction (adaptive vs static): 72.0%
cycle   0: lat=0.150 carb=0.550 priv=0.150 therm=0.150
cycle  69: lat=0.498 carb=0.352 priv=0.050 therm=0.101
cycle  90: lat=0.320 carb=0.436 priv=0.050 therm=0.194
cycle 119: lat=0.403 carb=0.429 priv=0.050 therm=0.118
cycle 140: lat=0.438 carb=0.347 priv=0.050 therm=0.164
cycle 159: lat=0.384 carb=0.431 priv=0.050 therm=0.135
cycle 199: lat=0.501 carb=0.337 priv=0.050 therm=0.112


## 3. Cross-Architecture Generalisation

Extends the cost model to three further architecture profiles — ResNet-50 as a larger CNN, ViT-Base as a Transformer, and a CLIP-style two-tower vision-language model as a multimodal case — reusing the same optimiser. Tests whether the interior-optimum split behaviour observed on MobileNetV2 generalises, or whether it depends on tensor size shrinking with network depth (a property CNNs have and Transformers largely do not).

In [3]:
import numpy as np

rng = np.random.default_rng(11)


def cnn_profile(L, total_local_ms, mean_power_w, input_kb, n_downsample,
                 total_mflops, priv_frac, seed):
    r = np.random.default_rng(seed)
    raw = np.linspace(1.6, 0.55, L)
    t_edge = raw / raw.sum() * total_local_ms
    p_edge = mean_power_w + r.normal(0, 0.15, L)
    tensor = np.zeros(L)
    stage_positions = set(int(round(p)) for p in
                           np.linspace(0, L - 1, n_downsample + 1)[1:])
    val = input_kb
    for i in range(L):
        if i in stage_positions:
            val /= 2.0
        tensor[i] = val
    mflops = np.linspace(1.6, 0.6, L)
    mflops = mflops / mflops.sum() * total_mflops
    priv_zone = set(range(max(1, int(round(L * priv_frac)))))
    return dict(name='cnn', L=L, t_edge_ms=t_edge, p_edge_w=p_edge,
                tensor_kb=tensor, layer_mflops=mflops, priv_zone=priv_zone)

def transformer_profile(L, total_local_ms, mean_power_w, seq_len, hidden_dim,
                          total_mflops, priv_frac, seed):
    r = np.random.default_rng(seed)
    t_edge = np.full(L, total_local_ms / L)
    p_edge = mean_power_w + r.normal(0, 0.15, L)
    tensor_bytes = seq_len * hidden_dim * 4
    tensor = np.full(L, tensor_bytes / 1024.0)
    mflops = np.full(L, total_mflops / L)
    priv_zone = set(range(max(1, int(round(L * priv_frac)))))
    return dict(name='transformer', L=L, t_edge_ms=t_edge, p_edge_w=p_edge,
                tensor_kb=tensor, layer_mflops=mflops, priv_zone=priv_zone)

def multimodal_profile(L_tower, L_fusion, total_tower_ms, total_fusion_ms,
                         mean_power_w, seq_len_v, seq_len_t, hidden_dim,
                         total_mflops_tower, total_mflops_fusion, priv_frac,
                         seed):
    r = np.random.default_rng(seed)
    L = L_tower + L_fusion
    t_tower_block = total_tower_ms / L_tower
    t_fusion_block = total_fusion_ms / L_fusion
    p_edge = mean_power_w + r.normal(0, 0.15, L)

    t_edge = np.zeros(L)
    tensor = np.zeros(L)
    mflops = np.zeros(L)
    tensor_v_block = (seq_len_v * hidden_dim * 4) / 1024.0
    tensor_t_block = (seq_len_t * hidden_dim * 4) / 1024.0
    for k in range(L_tower):
        t_edge[k] = t_tower_block
        tensor[k] = tensor_v_block + tensor_t_block
        mflops[k] = (total_mflops_tower / L_tower)
    for j in range(L_fusion):
        idx = L_tower + j
        t_edge[idx] = t_tower_block + t_fusion_block * (j + 1) / L_fusion * 0
    cum = np.zeros(L)
    running = 0.0
    for k in range(L_tower):
        running = t_tower_block * (k + 1)
        cum[k] = running
    tower_total = t_tower_block * L_tower
    for j in range(L_fusion):
        tower_total += t_fusion_block
        cum[L_tower + j] = tower_total
    t_edge_cum = cum

    tensor_fusion_block = ((seq_len_v + seq_len_t) * hidden_dim * 4) / 1024.0
    for j in range(L_fusion):
        tensor[L_tower + j] = tensor_fusion_block

    remaining_mflops = np.zeros(L)
    total_mflops = total_mflops_tower * 2 + total_mflops_fusion
    done_mflops = np.zeros(L)
    for k in range(L_tower):
        done_mflops[k] = (total_mflops_tower / L_tower) * (k + 1) * 2
    running_done = total_mflops_tower * 2
    for j in range(L_fusion):
        running_done += total_mflops_fusion / L_fusion
        done_mflops[L_tower + j] = running_done
    remaining_mflops = total_mflops - done_mflops

    priv_zone = set(range(max(1, int(round(L_tower * priv_frac)))))
    return dict(name='multimodal', L=L, t_edge_ms=None, t_edge_cum_ms=t_edge_cum,
                p_edge_w=p_edge, tensor_kb=tensor, remaining_mflops=remaining_mflops,
                priv_zone=priv_zone)

PROFILES = {
    'MobileNetV2 (small CNN)': cnn_profile(
        L=18, total_local_ms=450, mean_power_w=4.0, input_kb=588,
        n_downsample=8, total_mflops=300, priv_frac=0.22, seed=42),
    'ResNet-50 (large CNN)': cnn_profile(
        L=16, total_local_ms=1900, mean_power_w=5.5, input_kb=588,
        n_downsample=5, total_mflops=4100, priv_frac=0.20, seed=43),
    'ViT-Base (Transformer)': transformer_profile(
        L=12, total_local_ms=950, mean_power_w=5.0, seq_len=197,
        hidden_dim=768, total_mflops=17600, priv_frac=0.17, seed=44),
    'CLIP-style (Multimodal)': multimodal_profile(
        L_tower=6, L_fusion=4, total_tower_ms=420, total_fusion_ms=180,
        mean_power_w=5.2, seq_len_v=197, seq_len_t=77, hidden_dim=512,
        total_mflops_tower=4200, total_mflops_fusion=1800, priv_frac=0.20,
        seed=45),
}

def ma_edge_time_ms(profile, k):
    if profile['name'] == 'multimodal':
        return float(profile['t_edge_cum_ms'][k])
    return float(np.sum(profile['t_edge_ms'][:k + 1]))

def ma_edge_energy_J(profile, k):
    if profile['name'] == 'multimodal':
        return float(np.mean(profile['p_edge_w'][:k + 1])) * ma_edge_time_ms(profile, k) / 1000.0
    idx = slice(0, k + 1)
    return float(np.sum(profile['p_edge_w'][idx] * (profile['t_edge_ms'][idx] / 1000.0)))

def ma_p_compute(profile, k):
    return float(np.mean(profile['p_edge_w'][:k + 1]))

def remaining_mflops(profile, k):
    if profile['name'] == 'multimodal':
        return float(profile['remaining_mflops'][k])
    return float(np.sum(profile['layer_mflops'][k + 1:])) if k + 1 < profile['L'] else 0.0

def ma_uplink_time_ms(profile, k, region, bandwidth_mbps):
    bits = profile['tensor_kb'][k] * 1024 * 8
    return bits / (bandwidth_mbps * 1e6) * 1000.0 + region['rtt_ms'] / 2.0

def ma_net_energy_J(profile, k):
    bits = profile['tensor_kb'][k] * 1024 * 8
    return bits * E_per_bit_nJ * 1e-9

def ma_cloud_time_ms(profile, k, region):
    return remaining_mflops(profile, k) / (CLOUD_MFLOPS_PER_MS * region['compute_scale'])

def ma_cloud_energy_J(profile, k, region):
    return remaining_mflops(profile, k) * J_PER_MFLOP_CLOUD * PUE

def ma_latency_ms(profile, k, region_name, bandwidth_mbps):
    region = regions[region_name]
    return (ma_edge_time_ms(profile, k) + ma_uplink_time_ms(profile, k, region, bandwidth_mbps)
            + ma_cloud_time_ms(profile, k, region) + 2.0)

def ma_carbon_gco2(profile, k, region_name, bandwidth_mbps):
    region = regions[region_name]
    e = ma_edge_energy_J(profile, k) + ma_net_energy_J(profile, k) + ma_cloud_energy_J(profile, k, region)
    return (e / 3.6e6) * region['CI_now']

def ma_mutual_info(k):
    return np.exp(-0.35 * k)

def priv_penalty(profile, k):
    kmax = profile['L'] - 1
    i_max, i_min = ma_mutual_info(0), ma_mutual_info(kmax)
    return (ma_mutual_info(k) - i_min) / (i_max - i_min)

WEIGHTS = dict(lat=0.15, carb=0.55, priv=0.15, therm=0.0)

def ma_normalise(values):
    vmin, vmax = min(values), max(values)
    span = vmax - vmin
    return [0.0 for _ in values] if span < 1e-9 else [(v - vmin) / span for v in values]

def evaluate_architecture(profile, bandwidth_mbps=10.0):
    L = profile['L']
    priv_zone = profile['priv_zone']
    feasible = [(k, r) for k in range(L) if k not in priv_zone for r in regions]
    raw_lat = [ma_latency_ms(profile, k, r, bandwidth_mbps) for k, r in feasible]
    raw_carb = [ma_carbon_gco2(profile, k, r, bandwidth_mbps) for k, r in feasible]
    n_lat = ma_normalise(raw_lat)
    n_carb = ma_normalise(raw_carb)

    best, best_J = None, float('inf')
    for i, (k, r) in enumerate(feasible):
        priv = priv_penalty(profile, k)
        w = dict(WEIGHTS)
        s = w['lat'] + w['carb'] + w['priv']
        J = (w['lat']/s) * n_lat[i] + (w['carb']/s) * n_carb[i] + (w['priv']/s) * priv
        if J < best_J:
            best_J, best = J, (k, r, raw_lat[i], raw_carb[i])

    k_star, r_star, lat_star, carb_star = best
    local_lat = ma_edge_time_ms(profile, L - 1)
    local_carb = ma_carbon_gco2(profile, L - 1, 'IN-South', bandwidth_mbps)
    lat_star_best, r_fast = min(
        ((ma_latency_ms(profile, k, r, bandwidth_mbps), r) for k in range(L) for r in regions),
        key=lambda x: x[0])

    return dict(k_star=k_star, r_star=r_star, ma_latency_ms=lat_star, ma_carbon_gco2=carb_star,
                local_only_latency=local_lat, local_only_carbon=local_carb,
                latency_first_latency=lat_star_best,
                carbon_reduction_pct=100 * (1 - carb_star / local_carb),
                tensor_kb_at_kstar=profile['tensor_kb'][k_star],
                tensor_kb_range=(float(np.min(profile['tensor_kb'])), float(np.max(profile['tensor_kb']))))

if __name__ == '__main__':
    print(f"{'Architecture':<26}{'k*':>4}{'r*':>10}{'Lat(ms)':>9}{'CarbRed%':>10}{'TensorRange(KB)':>18}")
    for name, profile in PROFILES.items():
        res = evaluate_architecture(profile)
        tmin, tmax = res['tensor_kb_range']
        print(f"{name:<26}{res['k_star']:>4}{res['r_star']:>10}{res['ma_latency_ms']:>9.1f}"
              f"{res['carbon_reduction_pct']:>10.1f}{f'{tmin:.0f}-{tmax:.0f}':>18}")
        print(f"    local-only: {res['local_only_latency']:.1f} ms, "
              f"{res['local_only_carbon']:.4f} gCO2 | "
              f"latency-first best: {res['latency_first_latency']:.1f} ms")

Architecture                k*        r*  Lat(ms)  CarbRed%   TensorRange(KB)
MobileNetV2 (small CNN)      6   EU-West    382.8      87.2             2-588
    local-only: 450.0 ms, 0.0004 gCO2 | latency-first best: 302.8 ms
ResNet-50 (large CNN)        3   EU-West    998.8      91.1            18-588
    local-only: 1900.0 ms, 0.0021 gCO2 | latency-first best: 677.9 ms
ViT-Base (Transformer)       2   EU-West    840.0      91.2           591-591
    local-only: 950.0 ms, 0.0010 gCO2 | latency-first best: 607.6 ms
CLIP-style (Multimodal)      1   EU-West    695.7      92.1           548-548
    local-only: 600.0 ms, 0.0006 gCO2 | latency-first best: 548.5 ms


## 4. Comparison Against SOTA-Adjacent Baselines

Adds two further baselines, each a re-implementation of a design principle from prior work rather than a reproduction of that work's own published numbers: a Carbon-Aware Greedy baseline (route to the lowest-carbon-intensity region, then minimise latency) and a Privacy-Aware Static Split baseline (fix the split point once, at the shallowest layer that meets a fixed disentanglement threshold, and never revisit it). Evaluated under the same 50-trial randomised protocol and sustained-load thermal test as Section 1.

In [4]:
import numpy as np

rng = np.random.default_rng(99)
N_TRIALS = 50

PASS_MI_THRESHOLD = 0.15

def select_split_carbon_aware_greedy(bandwidth_mbps):
    r_star = min(regions, key=lambda r: regions[r]['CI_now'])
    k_star = min(range(L), key=lambda k: latency_ms(k, r_star, bandwidth_mbps))
    return k_star, r_star

def select_split_privacy_aware_static():
    for k in range(L):
        if P_priv(k) <= PASS_MI_THRESHOLD:
            k_star = k
            break
    else:
        k_star = L - 1
    r_star = min(regions, key=lambda r: regions[r]['rtt_ms'])
    return k_star, r_star

def select_split_proposed(bandwidth_mbps, t_device, t_amb, soc):
    k, r, _ = select_split_active(bandwidth_mbps, t_device, t_amb, soc)
    return k, r

STRATEGIES = ['local_only', 'latency_first', 'carbon_aware_greedy',
              'privacy_aware_static', 'proposed_active']

def decide(strategy, bandwidth, t_device, t_amb, soc):
    if strategy == 'local_only':
        return L - 1, None
    if strategy == 'latency_first':
        return select_split_latency_first(bandwidth)
    if strategy == 'carbon_aware_greedy':
        return select_split_carbon_aware_greedy(bandwidth)
    if strategy == 'privacy_aware_static':
        return select_split_privacy_aware_static()
    if strategy == 'proposed_active':
        return select_split_proposed(bandwidth, t_device, t_amb, soc)
    raise ValueError(strategy)

def run_experiment():
    results = {s: dict(carbon=[], latency=[], priv_violation=0, k_hist=[])
               for s in STRATEGIES}

    for trial in range(N_TRIALS):
        bandwidth = float(rng.lognormal(mean=np.log(10.0), sigma=0.5))
        t_init = float(rng.uniform(28, 36))
        t_amb = float(rng.uniform(22, 30))
        soc = float(rng.uniform(0.2, 1.0))

        for s in STRATEGIES:
            k, r = decide(s, bandwidth, t_init, t_amb, soc)
            r_for_cost = r if r is not None else 'IN-South'
            results[s]['carbon'].append(carbon_gco2(k, r_for_cost, bandwidth))
            results[s]['latency'].append(
                latency_ms(k, r_for_cost, bandwidth) if r is not None else edge_time_ms(k))
            results[s]['k_hist'].append(k)
            if k in PRIV_ZONE:
                results[s]['priv_violation'] += 1

    return results

def sustained_throttle(strategy, t_init, t_amb, soc, bandwidth):
    out = simulate_sustained(strategy if strategy in ('local_only', 'latency_first', 'proposed_active')
                                  else '_custom', t_init, t_amb, soc, bandwidth)
    return out

def sustained_throttle_generic(strategy, t_init, t_amb, soc, bandwidth, step_s=0.05):
    t_device = t_init
    elapsed = 0.0
    throttled = False
    max_temp = t_device
    duration_s = 60.0

    fixed_kr = None
    if strategy == 'latency_first':
        fixed_kr = select_split_latency_first(bandwidth)
    elif strategy == 'carbon_aware_greedy':
        fixed_kr = select_split_carbon_aware_greedy(bandwidth)
    elif strategy == 'privacy_aware_static':
        fixed_kr = select_split_privacy_aware_static()

    while elapsed < duration_s and not throttled:
        dvfs_mult = 1.0
        if strategy == 'local_only':
            k, r = L - 1, None
        elif strategy == 'proposed_active':
            k, r, dvfs_mult = select_split_active(bandwidth, t_device, t_amb, soc)
        else:
            k, r = fixed_kr

        r_for_cost = r if r is not None else 'IN-South'
        cycle_ms = latency_ms(k, r_for_cost, bandwidth) if r is not None else edge_time_ms(k)
        active_s = edge_time_ms(k) / 1000.0
        idle_s = max(0.0, cycle_ms / 1000.0 - active_s)
        active_power = p_compute(k) * dvfs_mult

        t_remaining = active_s
        while t_remaining > 0 and not throttled and elapsed < duration_s:
            dt = min(step_s, t_remaining)
            t_device = _integrate_step(t_device, soc, active_power, t_amb, dt)
            max_temp = max(max_temp, t_device)
            if t_device >= T_CRITICAL:
                throttled = True
            t_remaining -= dt
            elapsed += dt

        t_remaining = idle_s
        while t_remaining > 0 and not throttled and elapsed < duration_s:
            dt = min(step_s, t_remaining)
            t_device = _integrate_step(t_device, soc, P_IDLE_W, t_amb, dt)
            max_temp = max(max_temp, t_device)
            if t_device >= T_CRITICAL:
                throttled = True
            t_remaining -= dt
            elapsed += dt

    return throttled, max_temp

if __name__ == '__main__':
    res = run_experiment()
    local_carbon_ref = float(np.mean(res['local_only']['carbon']))

    throttle_rng = np.random.default_rng(99)
    throttle_counts = {s: 0 for s in STRATEGIES}
    for trial in range(N_TRIALS):
        bandwidth = float(throttle_rng.lognormal(mean=np.log(10.0), sigma=0.5))
        t_init = float(throttle_rng.uniform(28, 36))
        t_amb = float(throttle_rng.uniform(22, 30))
        soc = float(throttle_rng.uniform(0.2, 1.0))
        for s in STRATEGIES:
            throttled, _ = sustained_throttle_generic(s, t_init, t_amb, soc, bandwidth)
            throttle_counts[s] += int(throttled)

    print(f"{'Strategy':<22}{'MeanLat(ms)':>13}{'Carbon(norm%)':>15}{'PrivViol%':>11}{'ModeSplit':>11}{'Throttle%':>11}")
    for s in STRATEGIES:
        r = res[s]
        mean_lat = np.mean(r['latency'])
        norm_carbon = 100.0 * np.mean(r['carbon']) / local_carbon_ref
        priv_viol_pct = 100.0 * r['priv_violation'] / N_TRIALS
        mode_k = int(np.bincount(r['k_hist']).argmax())
        throttle_pct = 100.0 * throttle_counts[s] / N_TRIALS
        print(f"{s:<22}{mean_lat:>13.1f}{norm_carbon:>15.1f}{priv_viol_pct:>11.1f}{mode_k:>11d}{throttle_pct:>11.1f}")

Strategy                MeanLat(ms)  Carbon(norm%)  PrivViol%  ModeSplit  Throttle%
local_only                    450.0          100.0        0.0         17      100.0
latency_first                 297.4           45.5        8.0          4       98.0
carbon_aware_greedy           377.4           11.4        8.0          4       96.0
privacy_aware_static          309.0           51.3        0.0          6      100.0
proposed_active               385.6           11.7        0.0          4        4.0
